# Z.AI GLM on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Z.AI's GLM models. GLM 5.3 is the newest generation, and it is on `bedrock-runtime`
only, behind an inference profile, so it has its own notebook,
[`02-glm-5-3`](02-glm-5-3.ipynb). `zai.glm-5` is on both
endpoints and is the default for the sections before that. GLM-4.7 Flash is the
low-latency, low-cost option, which makes it a good first model to try before moving up
to the flagship.

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| GLM-5.3 | — | `us.zai.glm-5.3` | Newest; inference profile only, `global.zai.glm-5.3` where there is no `us.` profile ([`02-glm-5-3`](02-glm-5-3.ipynb)) |
| GLM-5 | `zai.glm-5` | `zai.glm-5` | On both endpoints; the default here |
| GLM-4.7 | `zai.glm-4.7` | `zai.glm-4.7` | Previous flagship |
| GLM-4.7 Flash | `zai.glm-4.7-flash` | `zai.glm-4.7-flash` | Low latency and cost |
| GLM-4.6 | `zai.glm-4.6` | — | Earlier generation, `bedrock-mantle` only |

IDs as of October 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path, except GLM 5.3, which `bedrock-mantle` does not carry. On
`bedrock-runtime` they serve **Converse** under the same IDs, except GLM-4.6, which is
absent there, and GLM 5.3, which needs its inference profile (section 7 and [`02-glm-5-3`](02-glm-5-3.ipynb)).

**Covered:** first call, streaming, multi-turn, reasoning, tool use, structured output,
and Converse on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check, a translation from a `bedrock-mantle` model ID to the `bedrock-runtime` form
(`runtime_id_for()`), and a lenient JSON parser.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient, resolve_runtime_id, runtime_id_for

REGION = "us-east-1"
GLM53 = "zai.glm-5.3"
GLM5 = "zai.glm-5"
GLM47 = "zai.glm-4.7"
FLASH = "zai.glm-4.7-flash"
GLM46 = "zai.glm-4.6"

# bedrock-mantle: the OpenAI SDK, with a short-term Bedrock API key minted from your
# AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

# The two endpoints do not agree on these IDs, so print the ID each one takes rather
# than a yes/no: GLM 5.3 is profile-only on bedrock-runtime and absent from
# bedrock-mantle, so a bare "runtime" would invite the bare ID, which is refused
# (02-glm-5-3). Resolve the runtime ID instead of reusing the string.
for model in (GLM53, GLM5, GLM47, FLASH, GLM46,):
    served = endpoints_for(model, REGION)
    mantle_id = model if served["mantle"] else "-"
    runtime_id = runtime_id_for(model, REGION) or "-"
    print(f"{model:19} mantle: {mantle_id:19} runtime: {runtime_id}")

zai.glm-5.3         mantle: -                   runtime: us.zai.glm-5.3
zai.glm-5           mantle: zai.glm-5           runtime: zai.glm-5
zai.glm-4.7         mantle: zai.glm-4.7         runtime: zai.glm-4.7
zai.glm-4.7-flash   mantle: zai.glm-4.7-flash   runtime: zai.glm-4.7-flash
zai.glm-4.6         mantle: zai.glm-4.6         runtime: -


## 1. First call

A single request with a short question. If the model spends the whole `max_tokens`
budget before it answers, `content` is `None` and `finish_reason` is `length`, so the
cell prints both.

In [2]:
response = client.chat.completions.create(
    model=GLM5,
    messages=[{"role": "user", "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=1000,
)
choice = response.choices[0]
answer = (choice.message.content or "").strip()
print(answer, "| Canberra:", "canberra" in answer.lower())
print(f"finish_reason={choice.finish_reason}  total_tokens={response.usage.total_tokens}")

Canberra | Canberra: True
finish_reason=stop  total_tokens=20


## 2. Streaming

Set `stream=True` and read `delta.content` from each chunk. Check that a chunk has
choices before indexing it, and read `finish_reason` from the last one: `stop` means the
answer is complete.

In [3]:
stream = client.chat.completions.create(
    model=GLM5,
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}],
    max_tokens=1000,
    stream=True,
)
finish_reason = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.choices and chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason
print(f"\nfinish_reason={finish_reason}")

1, 2, 3, 4, 5, 6, 7, 8, 9, 10


finish_reason=stop


## 3. Multi-turn

Chat Completions keeps no conversation state on the server, so each request carries the
whole history. Append the assistant's reply before the next user turn.

In [4]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."},
]
first = client.chat.completions.create(model=GLM5, messages=messages, max_tokens=1000)
messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.chat.completions.create(model=GLM5, messages=messages, max_tokens=1000)

print("turn 1:", (first.choices[0].message.content or "").strip())
print("turn 2:", (second.choices[0].message.content or "").strip())

turn 1: Noted.
turn 2: Teal.


## 4. Reasoning

Set `reasoning_effort` and GLM reasons before it answers. The reasoning comes back in
`choices[0].message.reasoning`, a field the OpenAI schema does not define. The SDK still
exposes it as an attribute, but code written against the published types will not see
it. Reasoning is billed as output and spends the budget before the answer starts, so
give it a larger `max_tokens`.

In [5]:
response = client.chat.completions.create(
    model=GLM5,
    messages=[{"role": "user", "content": 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.'}],
    reasoning_effort="high",
    max_tokens=4000,
)
message = response.choices[0].message
trace = getattr(message, "reasoning", None) or ""
answer = (message.content or "").strip()
print("answer   :", answer, "| 0.05 present:", "0.05" in answer)
print(f"reasoning: {len(trace)} characters")
print(f"finish_reason={response.choices[0].finish_reason}  "
      f"completion_tokens={response.usage.completion_tokens}")

answer   : $0.05 | 0.05 present: True
reasoning: 1290 characters
finish_reason=stop  completion_tokens=471


## 5. Tool use

A tool loop sends the tool definitions, runs each call the model makes, returns the
results, and repeats until the model answers in text. Check each call's arguments before
running it: `finish_reason == "tool_calls"` means the model wants a tool, and says
nothing about whether the arguments are usable.

In [6]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "type": "function",
    "function": {
        "name": "lookup_inventory",
        "description": "Current stock level for a SKU.",
        "parameters": {
            "type": "object",
            "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
            "required": ["sku"],
        },
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t["function"] for t in TOOLS if t["function"]["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [7]:
messages = [{"role": "user", "content": "How many units of SKU A-100 do we have?"}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = client.chat.completions.create(
        model=GLM5, messages=messages, tools=TOOLS, max_tokens=1500
    )
    message = response.choices[0].message
    if not message.tool_calls:
        break
    messages.append(message.model_dump(exclude_none=True))
    for call in message.tool_calls:
        args = parse_json_lenient(call.function.arguments)
        result = run_tool(call.function.name, args)
        print(f"tool call: {call.function.name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

answer = (message.content or "").strip()
print("answer:", answer)
# 42 is the quantity the stand-in tool reports for A-100, so the answer must carry it.
print("42 units reported:", "42" in answer)

tool call: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: You have **42 units** of SKU A-100 in stock.
42 units reported: True


## 6. Structured output

`response_format` with a JSON Schema asks the model for output in that shape. Check
`finish_reason` first, because a cut-off answer is cut-off JSON. Parse leniently,
because some models add text after a valid object. A schema controls the shape of the
output and has no say over the values in it, so the cell checks each value against the
source text.

In [8]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.chat.completions.create(
    model=GLM5,
    messages=[{"role": "user", "content": (
        "Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
        f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}")}],
    response_format={"type": "json_schema", "json_schema": {"name": "invoice", "strict": True, "schema": SCHEMA}},
    max_tokens=1500,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

invoice = parse_json_lenient(choice.message.content)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 7. On `bedrock-runtime`: Converse

GLM-5, GLM-4.7 and GLM-4.7 Flash are on `bedrock-runtime` under the same IDs; GLM-4.6 is
not, and GLM 5.3 needs its inference profile ([`02-glm-5-3`](02-glm-5-3.ipynb)). Converse has one request shape
for every provider: `content` is a list of blocks, and settings such as the token limit
go in `inferenceConfig`. An option only one provider understands, such as
`reasoning_effort`, goes in `additionalModelRequestFields`, which Converse passes to the
model unchecked. Reasoning arrives as a `reasoningContent` block when the model returns
one, and where it sits depends on the model: GLM-5 returns it ahead of the text, GLM 5.3
after it (October 2026). Read blocks by type instead of taking `content[0]`. The Converse
tool loop is in
[`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb).

In [9]:
response = runtime.converse(
    modelId="zai.glm-5",
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
    additionalModelRequestFields={"reasoning_effort": "high"},
)
blocks = response["output"]["message"]["content"]
text = "".join(b["text"] for b in blocks if "text" in b)
print("blocks    :", [next(iter(b)) for b in blocks])
print("answer    :", text.strip(), "| Canberra:", "canberra" in text.lower())
trace = "".join(b["reasoningContent"].get("reasoningText", {}).get("text", "") for b in blocks if "reasoningContent" in b)
print("reasoning :", len(trace), "characters")
print("stopReason:", response["stopReason"])

blocks    : ['reasoningContent', 'text']
answer    : Canberra | Canberra: True
reasoning : 295 characters
stopReason: end_turn


## Things that differ

What trips people up on this family, and what to do about it. Behaviour observed in
September and October 2026.

| Behaviour | What you see | What to do |
|---|---|---|
| Responses API on `bedrock-mantle` | `POST /v1/responses` returns 400 `The model '…' does not support the '/v1/responses' API` | Use Chat Completions for these models. GLM 5.3 serves the Responses API on `bedrock-runtime` ([`02-glm-5-3`](02-glm-5-3.ipynb)). |
| Responses API on `bedrock-runtime` | `zai.glm-5`, `zai.glm-4.7` and `zai.glm-4.7-flash` on `/openai/v1/responses` return 400 `The model '…' does not support the '/openai/v1/responses' API` | GLM 5.3 serves it ([`02-glm-5-3`](02-glm-5-3.ipynb)). For the others, use Chat Completions on `/openai/v1`, or Converse. |
| Path prefix | On `bedrock-mantle`, `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route``. On `bedrock-runtime` that path answers 200 for `zai.glm-5`, `zai.glm-4.7` and `zai.glm-4.7-flash` under their bare IDs, and 400 `The provided model identifier is invalid.` for `zai.glm-4.6` (October 2026) | Use the bare `/v1` path on `bedrock-mantle`. The prefix is not interchangeable between the endpoints: `bedrock-runtime` has an `/openai/v1` route of its own that serves the GLM models that endpoint carries, GLM 5.3 among them ([`02-glm-5-3`](02-glm-5-3.ipynb)), so the mantle 400 describes that route rather than the models. |
| Conversation state | No `previous_response_id` | Send the full `messages` list on every turn. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget and check `finish_reason` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Throttling is token-based and there is no requests-per-minute quota. Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Metrics | Nothing under `AWS/Bedrock` in CloudWatch | Mantle metrics are in the `AWS/BedrockMantle` namespace. |
| Numbers in structured output | Asked without a number format, GLM-5 returned `1` and `1280.5012812…` for `AUD 1,280.50` in two of five runs | Ask for plain decimals, as section 6 does, and check the value. |
| GLM-4.6 on `bedrock-runtime` | Not in the `bedrock-runtime` catalogue | Call it on `bedrock-mantle`, or move to a later GLM. |
| Where the reasoning block sits | GLM-5 returns `reasoningContent` before the text block on Converse, GLM 5.3 after it | Read content blocks by type. Indexing `content[0]` gets the trace for one model and the answer for the other. |

## Next

- GLM 5.3, on `bedrock-runtime` only:
  [`06-zai-glm/02`](02-glm-5-3.ipynb)
- Other families on Chat Completions:
  [`04-qwen/01`](../04-qwen/01-qwen3-core-and-tools.ipynb),
  [`05-deepseek/01`](../05-deepseek/01-deepseek-v3-reasoning.ipynb),
  [`09-minimax/01`](../09-minimax/01-minimax-m2.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)